Kylie Dennison and Blaine Aubrey 
Final Project
4/22/26
CSE-551

Research Question 1: In this project, we aim to investigate whether movie genre influences the film's finances using real-world data.
Research Question 2: In this project, we aim to investigate whether budget correlates with revenue using real-world data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import json

In [ ]:
credits=pd.read_csv("tmdb_5000_credits.csv")
df = pd.read_csv('tmdb_5000_movies.csv')
#We will not use any rows that do not have a revenue or budget as they will not be relevant to our research
df = df[(df['revenue'] > 0) & (df['budget'] > 0)]
#Create some new data for our plots later 
df['profit'] = df['revenue'] - df['budget']
df['ROI'] = (df['profit'] / df['budget']) * 100
#We will identify the outliers but we want to keep them because they represent the best of the best. 
Q1 = df['revenue'].quantile(0.25)
Q3 = df['revenue'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[df['revenue'] > (Q3 + 1.5 * IQR)]
#Gather the genres
def extract_genre(json_str):
    genres = json.loads(json_str)
    return genres[0]['name'] if genres else 'Unknown'
df['primary_genre'] = df['genres'].apply(extract_genre)

In [ ]:
#Bar Chart Research Question 1
genre_rev = df.groupby('primary_genre')['revenue'].mean().sort_values(ascending=False)
sns.barplot(x=genre_rev.values, y=genre_rev.index, palette='nipy_spectral')
plt.title('The average revenue by single genre')
plt.xlabel('Avg Rev.(USD)')
plt.ylabel('Genre')
plt.show()

Bar Chart Interpretation:
This visualization shows which genres bring in the most most revenue on average. This is important because it shows where the most money making potential lies based purely on the genre. The key insight is that while many genres like action and drama have more entries and typically preform very well, the best preformers on avg are actually more family friendly movies like animated films. 

In [ ]:
#Line Plot Research Question 1
df['release_date'] = pd.to_datetime(df['release_date'])
df['year'] = df['release_date'].dt.year
yearly_rev = df.groupby('year')['revenue'].sum()
plt.plot(yearly_rev.index, yearly_rev.values, color='purple', marker='o')
plt.title('Total annual movie revenue')
plt.xlabel('Year')
plt.ylabel('Total Revenue in Billions')
plt.show()

Line Plot: This visualization shows the upward trend of revenue based on time. This is important because it allows us to see that as time progress we can see the film market growing, the only real exception to this occured around COVID times where we see the steep drop. The key insight is that the peak was around the 2010's; this means that the movies and their genres being made around this time were what we should be striving to make.

In [ ]:
#Histogram Research Question 1
sns.histplot(df['budget'], bins=30, kde=True, color='red')
plt.title('Distribution of movie budgets')
plt.xlabel('Budget in USD')
plt.ylabel('Frequency')
plt.show()

Histogram: This visualization shows a very large right skewed distribution. This is important because it shows us that most films are made with a relatively low budget. The key insight is that most movies are made with a low budget which must mean budget is not the only key to making a sucessful movie.

In [ ]:
#Scatter plot Research Question 1
sns.scatterplot(data=df, x='budget', y='revenue', alpha=0.5, color='purple')
plt.title('Relationship between budget and revenue')
plt.xlabel('Budget')
plt.ylabel('Revenue')
plt.show()

Scatter Plot: This visualization shows a strong correlation between the money spent and the money earned. This is important because for the most part it means that if a studio spends money they will earn a lot of money. The key insight is that there is noticeable variance, some low budget movies make a lot of money and some high budget movies make very little.

In [ ]:
#Box Plot Research Question 1
top_5 = df['primary_genre'].value_counts().nlargest(5).index
df_top5 = df[df['primary_genre'].isin(top_5)]
sns.boxplot(x='primary_genre', y='revenue', data=df_top5)
plt.title('Revenue spread across the top 5 genres')
plt.xlabel('Genre')
plt.show()

Box Plot: This visualization shows the median revenue with outliers based on genre. This is important because we can see the range and therefore the risk associted with each genre; such as horror having the most risk due to a low median and very few outliers. The key insight is that action movies have the most high earning outliers.

In [ ]:
#Multi Line Plot Research Question 1
top3 = ['Action','Drama','Comedy']
df_top3 = df[df['primary_genre'].isin(top3)].groupby(['year', 'primary_genre'])['revenue'].mean().unstack()
df_top3.plot()
plt.title('Avergae revenue change for top 3 genres')
plt.ylabel('Avg revenue')
plt.show()

Multi Line Plot: This visualization shows how the average earnings of top genres has shifted over time. This is important because it tracks the changing audience prefrences. The key insight is that action is now the leading genre in terms of revenue.

In [ ]:
#Faceted Research Question 1
gen = sns.FacetGrid(df_top5, col="primary_genre", col_wrap=3)
gen.map(sns.scatterplot, "popularity", "revenue")
plt.subplots_adjust(top=0.9)
gen.fig.suptitle('Revenue vs Popularity across genres')
plt.show()

Faceted: This visualization shows the relationship between popularity and revenue. This is important because it shows that if something is more popular it earns more money. The key insight is that the correlation is consistent no matter the genre.



In [ ]:
#Heatmap Research Question 1
cor = df[['budget','revenue','runtime','vote_average','popularity', 'profit']].corr()
sns.heatmap(cor, annot=True, cmap='RdBu_r')
plt.title('Correlation of Movie Metrics')
plt.show()

Heatmap: This visualization shows which variables are linked to each other. This is important because we can see which factors will influence the desired factor. The key insight is that budget and popularity are the best predictors of revenue and thus profit.

In [ ]:
#Annotated Research Question 1
top_mov = df.loc[df['profit'].idxmax()]
sns.scatterplot(data=df, x='budget', y='revenue')
plt.axhline (1e9, color='red')
plt.text(0,1.1e9, 'The Billion Dollar Club', color='red')
plt.annotate(f"Highest Profit: {top_mov['original_title']}", 
             xy=(top_mov['budget'], top_mov['revenue']),
             xytext=(top_mov['budget']+1e8, top_mov['revenue']),
             arrowprops=dict(facecolor='black', shrink=0.05))
plt.title('Budget vs Revenue with top preformers')
plt.show()

Annotated: This visualization shows the most sucuessful film and how few others cross the huge billion dollar threashold. This is important because it tells us what the gold standard is in terms of earnings. The key insight is how far the outliers sit above the normal films.

In [ ]:
#Custom Research Question 1
sns.violinplot(x='primary_genre', y='profit',data=df_top5, inner="quartile")
plt.title('Profit density distribution by genre')
plt.xlabel('genre')
plt.ylim(-1e8, 1e9)
plt.show()

Custom: This visulatization shows the probability density of profit for diffrent genres. This is important because we can see that most films actually fall in the widest part. The key insight is that drama has a high peak but also a wider body which would mean it is the safest genre to make a film in terms of making a profit.

In [ ]:
#Interactive Research Question 1
plot = px.scatter(df, x="budget", y="revenue", color="primary_genre", hover_name="original_title", size="popularity",
                  title="Interactive: Budget, Revenue, and Popularity")
plot.update_layout(xaxis_title="Budget", yaxis_title="Revenue")
plot.show()

This visulaization shows Budget, Revenue, Genre, and popularity, and title all in one. This is important because it allows viewers to identify specfic movies to see how they preformed. The key insight is that we can idenifty specifc movies to try and then analyze what makes them stand out.